# Evaluate DAN — Standalone Evaluation
Load pretrained DAN weights từ `outputs/models/best_dan_model.pth`, chạy inference + đánh giá.

In [ ]:
import os, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CLASS_NAMES = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
NUM_CLASSES = 7
BATCH_SIZE = 32
SAVE_DIR = os.path.join('outputs', 'evaluation')
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. RafDataSet — Load dữ liệu test

In [ ]:
class RafDataset(Dataset):
    def __init__(self, data_path, train=False, transform=None):
        self.transform = transform
        import pandas as pd
        csv_name = 'train_labels.csv' if train else 'test_labels.csv'
        df = pd.read_csv(os.path.join(data_path, csv_name))
        file_names = df['image'].values
        labels = df['label'].values
        self.target = np.array(labels - 1)
        split = 'train' if train else 'test'
        self.file_paths = [
            os.path.join(data_path, 'DATASET', split, str(lbl), fname)
            for fname, lbl in zip(file_names, labels)]
        print(f'  Loaded {split}: {len(self.file_paths)} samples')

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.file_paths[idx])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(img.copy())
        return img, self.target[idx]

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Loading test dataset...')
test_dataset = RafDataset('data', train=False, transform=val_tf)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Test batches: {len(test_loader)}')

## 2. Định nghĩa mô hình DAN

In [ ]:
class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super(DAN, self).__init__()
        from torchvision import models
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)

    def forward(self, x):
        x = self.features(x)
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        weighted_features = (x_flat * att_flat).sum(dim=-1)
        final_features = weighted_features.mean(dim=1)
        out = self.fc(final_features)
        out = self.bn(out)
        return out

## 3. Load pretrained weights

In [ ]:
CKPT_PATH = os.path.join('outputs', 'models', 'best_dan_model.pth')

print('Building DAN model...')
model = DAN(num_class=NUM_CLASSES, num_head=4)

if os.path.exists(CKPT_PATH):
    state = torch.load(CKPT_PATH, map_location='cpu')
    # DAN model weights are stored as pure state_dict
    if any(k.startswith('module.') for k in state):
        state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    print(f'Loaded: {CKPT_PATH}')
else:
    print(f'Không tìm thấy {CKPT_PATH}')

model = model.to(device)
model.eval()
print('Model ready!')

## 4. Inference

In [ ]:
print('Running inference...')
all_preds, all_labels, all_probs = [], [], []
start = time()

with torch.no_grad():
    for imgs, tgts in test_loader:
        logits = model(imgs.to(device))
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(tgts.tolist())
        all_probs.extend(probs.cpu().numpy())

P = np.array(all_preds)
T = np.array(all_labels)
probs = np.array(all_probs)
elapsed = time() - start
print(f'Done! {len(T)} samples in {elapsed:.1f}s ({len(T)/elapsed:.1f} img/s)')

## 5. Kết quả

In [ ]:
overall_acc = (P == T).sum() / len(T) * 100
print(f'Overall Accuracy: {overall_acc:.2f}%')
print()

per_class_acc = []
print('Per-Class Accuracy:')
for i in range(NUM_CLASSES):
    mask = (T == i)
    acc = (P[mask] == i).sum() / mask.sum() * 100
    per_class_acc.append(acc)
    print(f'  {CLASS_NAMES[i]:12s}: {acc:.2f}%')
print(f'  {"Mean":12s}: {np.mean(per_class_acc):.2f}%')
print()
print('Classification Report:')
print(classification_report(T, P, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(T, P)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[0])
ax[0].set_title('Confusion Matrix (Counts)')

cm_n = cm / cm.sum(1, keepdims=True) * 100
sns.heatmap(cm_n, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1])
ax[1].set_title('Confusion Matrix (%)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_dan.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve((T == i).astype(int), probs[:, i])
    auc_score = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{CLASS_NAMES[i]} (AUC={auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — DAN')
ax.legend(loc='lower right')
plt.savefig(os.path.join(SAVE_DIR, 'roc_curves_dan.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
errors = np.where(P != T)[0]
print(f'Total errors: {len(errors)}/{len(T)} ({len(errors)/len(T)*100:.1f}%)')

from collections import Counter
confused = [(CLASS_NAMES[T[i]], CLASS_NAMES[P[i]]) for i in errors]
print('\nTop confused pairs:')
for (true, pred), count in Counter(confused).most_common(8):
    print(f'  {true:12s} → {pred:12s}: {count}')

print('\nDAN Results Summary:')
print(f'  Overall Accuracy: {overall_acc:.2f}%')
print(f'  Mean Class Acc:  {np.mean(per_class_acc):.2f}%')
print(f'\nResults saved to: {SAVE_DIR}')